# Phase 1 — ogbn-arxiv Dataset Loading & EDA

This notebook implements Phase 1 from `IMPLEMENTATION_GUIDE.md`:
- Dataset loading and split stats
- Class distribution analysis
- `cs.HC` deep dive (split counts, entropy, degree)
- Cross-domain citation ratio analysis
- Export to `results/dataset_stats.json`

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import torch

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.eda import (
    compute_cross_domain_ratio,
    compute_degree_vectors,
    compute_per_class_neighbor_entropy,
    find_class_id,
    load_dataset,
    save_dataset_stats,
    split_class_counts,
    summarize_degree_stats,
    summarize_phase1,
)

sns.set_theme(style="whitegrid")

In [ ]:
DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
PLOTS_DIR = RESULTS_DIR / "eda"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

data, split_idx, dataset, label_names = load_dataset(DATA_ROOT)
labels = data.y.view(-1).to(torch.long)

num_nodes = int(data.num_nodes)
num_edges = int(data.edge_index.size(1))
num_features = int(data.x.size(1))
num_classes = int(labels.max().item() + 1)

print(f"Nodes: {num_nodes:,}")
print(f"Edges: {num_edges:,}")
print(f"Feature dim: {num_features}")
print(f"Classes: {num_classes}")
print(f"Train split: {split_idx['train'].numel():,}")
print(f"Valid split: {split_idx['valid'].numel():,}")
print(f"Test split: {split_idx['test'].numel():,}")

In [ ]:
class_counts = torch.bincount(labels, minlength=num_classes).cpu().numpy()
class_ids = list(range(num_classes))
class_names = [label_names.get(i, f"class_{i}") for i in class_ids]

cs_hc_id = find_class_id(label_names, "cs.HC")
colors = ["tab:red" if i == cs_hc_id else "tab:blue" for i in class_ids]

plt.figure(figsize=(16, 5))
plt.bar(range(num_classes), class_counts, color=colors)
plt.xticks(range(num_classes), class_names, rotation=75, ha="right", fontsize=8)
plt.title("ogbn-arxiv class distribution (40 classes)")
plt.ylabel("Number of nodes")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "class_distribution.png", dpi=200)
plt.show()

imbalance_ratio = float(class_counts.max() / max(class_counts.min(), 1))
print(f"Imbalance ratio (max/min class count): {imbalance_ratio:.2f}x")

In [ ]:
in_deg, out_deg, undeg = compute_degree_vectors(data)

if cs_hc_id is None:
    raise RuntimeError("Could not locate cs.HC class id in label mapping.")

cs_hc_mask = labels == cs_hc_id
cs_hc_split = split_class_counts(labels, split_idx, cs_hc_id)

print(f"cs.HC class id: {cs_hc_id}")
print("cs.HC split counts:", cs_hc_split)
print("\nDegree summary (undirected):")
print("  cs.HC:", summarize_degree_stats(undeg, cs_hc_mask))
print("  All nodes:", summarize_degree_stats(undeg))

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(undeg.float().cpu().numpy(), bins=100, stat="density", color="tab:blue", alpha=0.35, label="All nodes")
sns.histplot(undeg[cs_hc_mask].float().cpu().numpy(), bins=100, stat="density", color="tab:red", alpha=0.45, label="cs.HC")
plt.xlim(0, 200)
plt.title("Undirected degree distribution: cs.HC vs dataset")
plt.xlabel("Degree")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "degree_distribution_cs_hc_vs_all.png", dpi=200)
plt.show()

In [ ]:
per_class_entropy = compute_per_class_neighbor_entropy(data, labels)
sorted_entropy = sorted(per_class_entropy.items(), key=lambda x: x[1], reverse=True)

entropy_names = [label_names.get(c, f"class_{c}") for c, _ in sorted_entropy]
entropy_values = [v for _, v in sorted_entropy]
entropy_colors = ["tab:red" if c == cs_hc_id else "tab:blue" for c, _ in sorted_entropy]

plt.figure(figsize=(16, 5))
plt.bar(range(num_classes), entropy_values, color=entropy_colors)
plt.xticks(range(num_classes), entropy_names, rotation=75, ha="right", fontsize=8)
plt.title("Neighbor label entropy by class (higher = more interdisciplinary neighborhood)")
plt.ylabel("Entropy")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "neighbor_label_entropy_by_class.png", dpi=200)
plt.show()

print(f"cs.HC neighbor-label entropy: {per_class_entropy[cs_hc_id]:.4f}")

In [ ]:
cross_ratio = compute_cross_domain_ratio(data, labels)
sorted_ratio = sorted(cross_ratio.items(), key=lambda x: x[1], reverse=True)

ratio_classes = [c for c, _ in sorted_ratio]
ratio_names = [label_names.get(c, f"class_{c}") for c in ratio_classes]
ratio_values = [v for _, v in sorted_ratio]

highlight_ids = set()
for lbl in ["cs.HC", "cs.DS", "cs.CR"]:
    idx = find_class_id(label_names, lbl)
    if idx is not None:
        highlight_ids.add(idx)

ratio_colors = ["tab:red" if c in highlight_ids else "tab:blue" for c in ratio_classes]

plt.figure(figsize=(16, 5))
plt.bar(range(num_classes), ratio_values, color=ratio_colors)
plt.xticks(range(num_classes), ratio_names, rotation=75, ha="right", fontsize=8)
plt.title("Cross-domain citation ratio by class (sorted)")
plt.ylabel("Cross-domain citation ratio")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "cross_domain_citation_ratio_by_class.png", dpi=200)
plt.show()

for lbl in ["cs.HC", "cs.DS", "cs.CR"]:
    idx = find_class_id(label_names, lbl)
    if idx is not None:
        print(f"{lbl} (class {idx}): {cross_ratio[idx]:.4f}")

In [ ]:
summary = summarize_phase1(data, split_idx, label_names, cs_hc_label="cs.HC")
output_json = RESULTS_DIR / "dataset_stats.json"
save_dataset_stats(summary, output_json)

print(f"Saved summary JSON: {output_json}")
print(json.dumps({
    "num_nodes": summary["num_nodes"],
    "num_edges": summary["num_edges"],
    "num_classes": summary["num_classes"],
    "cs_hc": summary.get("cs_hc", {}),
}, indent=2)[:1200])